In [4]:
!pip install sentence-transformers rouge-score nltk
!python -m nltk.downloader punkt


Defaulting to user installation because normal site-packages is not writeable
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 1.5/1.5 MB 15.9 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=25024 sha256=59340d001ddd4093492160b874c63f468bc9a21a7848e7c35e28929155417446
  Stored in directory: c:\users\hp\appdata\local\pip\cache\wheels\85\9d\af\01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.2 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
<frozen runpy>:128: RuntimeWarning: 'nltk.downloader' found in sys.modules after import of package 'nltk', but prior to execution of 'nltk.downloader'; this may result in unpredictable behaviour
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


In [5]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from nltk.translate.bleu_score import sentence_bleu
from rouge_score import rouge_scorer
import ast
import nltk

nltk.download('punkt')

# Step 1: Load your file
df = pd.read_csv(r"C:\Users\hp\Downloads\recursive_chunks_with_embeddings.csv")

# Step 2: Parse embeddings (from strings to lists)
df["embedding"] = df["embedding"].apply(ast.literal_eval)

texts = df["text"].tolist()
embeddings = df["embedding"].tolist()

# Step 3: Initialize metric tools
bleu_scores = []
rouge1_scores = []
rougel_scores = []
cosine_scores = []

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

# Step 4: Compare each chunk with the previous one (skip 0th)
for i in range(1, len(texts)):
    # Cosine Similarity
    cos = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
    cosine_scores.append(cos)

    # BLEU Score
    bleu = sentence_bleu([texts[i-1].split()], texts[i].split())
    bleu_scores.append(bleu)

    # ROUGE Scores
    rouge = scorer.score(texts[i-1], texts[i])
    rouge1_scores.append(rouge["rouge1"].fmeasure)
    rougel_scores.append(rouge["rougeL"].fmeasure)

# Step 5: Print average scores
print("\n✅ Self-Comparison Metric Results (Your Recursive Chunks Only):")
print(f"Average Cosine Similarity: {sum(cosine_scores)/len(cosine_scores):.4f}")
print(f"Average BLEU Score:        {sum(bleu_scores)/len(bleu_scores):.4f}")
print(f"Average ROUGE-1 Score:     {sum(rouge1_scores)/len(rouge1_scores):.4f}")
print(f"Average ROUGE-L Score:     {sum(rougel_scores)/len(rougel_scores):.4f}")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
C:\Users\hp\AppData\Roaming\Python\Python312\site-packages\nltk\translate\bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 


✅ Self-Comparison Metric Results (Your Recursive Chunks Only):
Average Cosine Similarity: 0.6055
Average BLEU Score:        0.1377
Average ROUGE-1 Score:     0.3687
Average ROUGE-L Score:     0.1653
